In [11]:
import copy
import random
# 1. นำเข้าเฉพาะคลาสที่มีใน agents.py
from agents import Agent, GraphicEnvironment, Direction, Bump, Thing

# 2. นิยาม Food และ Water ขึ้นมาเอง (ป้องกัน ImportError)
class Food(Thing):
    pass

class Water(Thing):
    pass

# ==========================================
# 1. นิยาม SmartBlindDog Agent
# ==========================================
class SmartBlindDog(Agent):
    def __init__(self, program=None):
        super().__init__(program)
        self.location = [0, 0]
        self.direction = Direction("down")
        self.visited = set()            # เก็บพิกัด (x, y) ที่เคยเดินผ่าน
        self.visited.add((0, 0))        # บันทึกจุดเริ่มต้น

    def moveforward(self, success=True):
        if not success:
            return
        if self.direction.direction == Direction.R:
            self.location[0] += 1
        elif self.direction.direction == Direction.L:
            self.location[0] -= 1
        elif self.direction.direction == Direction.D:
            self.location[1] += 1
        elif self.direction.direction == Direction.U:
            self.location[1] -= 1
            
        # บันทึกตำแหน่งใหม่ลงใน Memory
        self.visited.add(tuple(self.location))

    def turn(self, d):
        self.direction = self.direction + d

    def eat(self, thing):
        if isinstance(thing, Food):
            print(f"🐶 SmartBlindDog: Ate Food at {self.location}")
            return True
        return False

    def drink(self, thing):
        if isinstance(thing, Water):
            print(f"🐶 SmartBlindDog: Drank Water at {self.location}")
            return True
        return False

# ==========================================
# 2. Smart Program Function (สมองของ Agent)
# ==========================================
def smart_dog_program(percepts):
    """
    Program ที่ใช้ Memory และ State ในการตัดสินใจเดิน
    """
    # [1] ลำดับความสำคัญแรก: หากพบอาหารหรือน้ำ ให้จัดการทันที
    for p in percepts:
        if isinstance(p, Food):
            return 'eat'
        elif isinstance(p, Water):
            return 'drink'

    # [2] อ่านสถานะ Bump (กำแพงข้างหน้า) และดึงอ็อบเจกต์ agent ออกมาจาก percepts
    has_bump = any(isinstance(p, Bump) for p in percepts)
    
    # ค้นหาอ็อบเจกต์ SmartBlindDog เพื่อเข้าถึง State (location, direction, visited)
    agent = None
    for p in percepts:
        if isinstance(p, SmartBlindDog):
            agent = p
            break

    # กรณี fallback สำรองถ้าดึงตัว agent ไม่เจอ ให้ใช้การสุ่มเบื้องต้น
    if agent is None:
        if has_bump:
            return random.choice(['turnright', 'turnleft'])
        return random.choice(['moveforward', 'turnright', 'turnleft'])

    # Helper function สำหรับคำนวณทิศทาง
    def get_ahead_location(loc, direction_str):
        x, y = loc[0], loc[1]
        if direction_str == Direction.R: return (x + 1, y)
        if direction_str == Direction.L: return (x - 1, y)
        if direction_str == Direction.D: return (x, y + 1)
        if direction_str == Direction.U: return (x, y - 1)
        return (x, y)

    curr_dir = agent.direction.direction
    right_dir = (agent.direction + Direction.R).direction
    left_dir = (agent.direction + Direction.L).direction

    ahead_loc = get_ahead_location(agent.location, curr_dir)
    right_loc = get_ahead_location(agent.location, right_dir)
    left_loc = get_ahead_location(agent.location, left_dir)

    # [3] ประเมินทางเลือก
    # - ข้างหน้าเดินได้ไหม? (ต้องไม่ชนกำแพง และยังไม่เคยเดินผ่าน)
    forward_is_unvisited = (not has_bump) and (ahead_loc not in agent.visited)
    right_is_unvisited = right_loc not in agent.visited
    left_is_unvisited = left_loc not in agent.visited

    # [4] ตัดสินใจเลือก Action ตาม Memory
    if forward_is_unvisited:
        return 'moveforward'
    
    # ถ้าด้านหน้าเคยไปแล้ว ลองดูขวาหรือซ้ายที่ยังไม่เคยไป
    unvisited_turns = []
    if right_is_unvisited:
        unvisited_turns.append('turnright')
    if left_is_unvisited:
        unvisited_turns.append('turnleft')

    if unvisited_turns:
        return random.choice(unvisited_turns)

    # [5] กรณีรอบตัวเคยไปหมดแล้ว (Dead end / Explored area) -> Backtrack/สุ่มทิศทางใหม่
    if has_bump:
        return random.choice(['turnright', 'turnleft'])
    else:
        return random.choice(['moveforward', 'turnright', 'turnleft'])

# ==========================================
# 3. Environment & Graphic System
# ==========================================
class SmartPark2D(GraphicEnvironment):
    def percept(self, agent):
        # ส่งค่าสิ่งที่อยู่ในพื้นที่ + ตัว agent เอง + Bump (ถ้ามี)
        things = self.list_things_at(agent.location)
        things.append(agent)  # ส่ง agent เข้าไปเพื่อให้อ่านค่า visited
        
        loc = copy.deepcopy(agent.location)
        if agent.direction.direction == Direction.R: loc[0] += 1
        elif agent.direction.direction == Direction.L: loc[0] -= 1
        elif agent.direction.direction == Direction.D: loc[1] += 1
        elif agent.direction.direction == Direction.U: loc[1] -= 1
            
        if not self.is_inbounds(loc):
            things.append(Bump())
        return things
    
    def execute_action(self, agent, action):
        if action == 'turnright':
            agent.turn(Direction.R)
        elif action == 'turnleft':
            agent.turn(Direction.L)
        elif action == 'moveforward':
            agent.moveforward()
        elif action == "eat":
            items = self.list_things_at(agent.location, tclass=Food)
            if len(items) != 0:
                if agent.eat(items[0]):
                    self.delete_thing(items[0])
        elif action == "drink":
            items = self.list_things_at(agent.location, tclass=Water)
            if len(items) != 0:
                if agent.drink(items[0]):
                    self.delete_thing(items[0])

# ==========================================
# 4. Run Simulation Demo
# ==========================================
if __name__ == "__main__":
    # สร้างสภาพแวดล้อมขนาด 5x5
    park = SmartPark2D(5, 5, color={
        'SmartBlindDog': (200, 0, 0),    # สีแดง
        'Water': (0, 200, 200),         # สีฟ้า
        'Food': (230, 115, 40)          # สีส้ม
    })

    # สร้าง Agent และสิ่งของวางในสวน
    dog = SmartBlindDog(smart_dog_program)
    park.add_thing(dog, [0, 0])
    park.add_thing(Food(), [1, 2])
    park.add_thing(Water(), [0, 1])
    park.add_thing(Water(), [2, 4])
    park.add_thing(Food(), [4, 3])

    print("🚀 เริ่มการจำลอง SmartBlindDog Simulation...")
    park.run(30)
    
    print("\n📍 สรุปพิกัดทั้งหมดที่ SmartBlindDog เคยสำรวจผ่าน (Memory set):")
    print(sorted(list(dog.visited)))

,,,,
,,,,
,,,,
,,,,
,,,,



📍 สรุปพิกัดทั้งหมดที่ SmartBlindDog เคยสำรวจผ่าน (Memory set):
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)]


In [ ]:
%pip install ipythonblocks